## Ejercicio 6: Optimización — Evitar múltiples `split()`

> **Nota sobre el dataset:** El archivo `persons.csv` real del repositorio tiene el esquema `name,age,gender,salary` (4 columnas) en lugar de `name,age,city`. La optimización se aplicó al esquema real extrayendo todos los campos en una sola pasada.

### 1. Problema de eficiencia del código original
En el código original se aplican transformaciones `map()` independientes sobre el RDD crudo, invocando `x.split(",")` varias veces por cada línea (una vez por columna). Al ejecutar acciones sobre cada columna derivada, Spark escanea repetidamente las mismas cadenas y crea múltiples listas temporales en memoria por cada registro.

### 2. Por qué la solución optimizada es más eficiente
Al ejecutar `split(",")` **una sola vez por línea** y almacenar la tupla `(nombre, edad, genero, salario)` dentro de un único RDD con `.cache()`:
- Se eliminan las llamadas redundantes a `split()` (reduciendo el trabajo de parseo a una sola pasada por registro).
- Se evita releer el archivo desde disco cuando se consultan distintas columnas.
- Se reduce el uso de CPU y la presión sobre el recolector de basura de Python, lo cual marca una diferencia notable con millones de registros.

In [1]:
import os, cloudpickle, pyspark.serializers
pyspark.serializers.cloudpickle = cloudpickle
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ejercicio6_OptimizacionSplit").getOrCreate()
sc = spark.sparkContext

ruta_csv = "file://" + os.path.abspath("./DATOS/persons.csv")
rdd = sc.textFile(ruta_csv)

# Quitar encabezado
encabezado = rdd.first()
rdd_datos = rdd.filter(lambda linea: linea != encabezado and linea.strip() != "")

# OPTIMIZADO: split() se ejecuta UNA sola vez por línea en una sola pasada
def parsear_persona(linea):
    try:
        partes = linea.strip().split(",")
        return (
            partes[0].strip(),        # name
            int(partes[1].strip()),   # age
            partes[2].strip(),        # gender
            float(partes[3].strip())  # salary
        )
    except (ValueError, IndexError):
        return None

rdd_personas = rdd_datos.map(parsear_persona).filter(lambda x: x is not None).cache()

# Extracción de columnas indexando la tupla ya parseada en O(1)
nombres = rdd_personas.map(lambda x: x[0])
edades = rdd_personas.map(lambda x: x[1])
generos = rdd_personas.map(lambda x: x[2])
salarios = rdd_personas.map(lambda x: x[3])

print("Registros parseados en una sola pasada (muestra):")
for registro in rdd_personas.take(5):
    print(registro)

print("\nNombres (muestra) :", nombres.take(5))
print("Edades (muestra)  :", edades.take(5))
print("Géneros (muestra) :", generos.take(5))
print("Salarios (muestra):", salarios.take(5))

spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/20 02:50:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Registros parseados en una sola pasada (muestra):


('John Doe', 30, 'Male', 50000.0)
('Jane Smith', 25, 'Female', 45000.0)
('David Johnson', 35, 'Male', 60000.0)
('Emily Davis', 28, 'Female', 52000.0)
('Michael Wilson', 40, 'Male', 75000.0)

Nombres (muestra) : ['John Doe', 'Jane Smith', 'David Johnson', 'Emily Davis', 'Michael Wilson']
Edades (muestra)  : [30, 25, 35, 28, 40]
Géneros (muestra) : ['Male', 'Female', 'Male', 'Female', 'Male']
Salarios (muestra): [50000.0, 45000.0, 60000.0, 52000.0, 75000.0]
